# NumPy 常用统计函数：用数据说话的艺术

> 这是「数据分析从入门到精通」系列的第 5 篇。前四篇我们把 NumPy 的基础操作都摸了一遍，这篇来聊聊数据分析的"基本功"——统计函数。毕竟，**不会算统计指标，就像萧何不会算账，再厉害的统筹能力也是白搭**。

---

嗨，我是小荷。

说实话，写这篇的时候我一直在想：萧何当年管着整个大汉的粮草账目，他是怎么做到心中有数的？我猜他肯定有一套自己的"统计方法"——哪些粮仓充盈、哪些紧缺、平均每个士兵消耗多少，这些数据他心里门儿清。

我们做数据分析也一样。**拿到一堆原始数据，第一步永远是算统计指标**——均值多少、波动大不大、有没有异常值。这些数字就像数据的"体检报告"，让你一眼看清状况。

NumPy 的统计函数，就是帮你快速出这份体检报告的工具。

---

## 集中趋势：数据往哪儿扎堆？

### 均值（mean）

最直观的"平均水平"，但有个致命弱点——**容易被极端值带偏**。

In [1]:
import numpy as np

salaries = np.array([8000, 8500, 9000, 9200, 150000])  # 最后一个是高管
print(f"平均工资: {salaries.mean():.0f}")  # 36840，明显失真


平均工资: 36940


### 中位数（median）

排序后中间那个数，**不受极端值影响**，收入数据用它更靠谱。

In [2]:
print(f"工资中位数: {np.median(salaries):.0f}")  # 9000，真实多了


工资中位数: 9000


### 众数（mode）

出现次数最多的值。NumPy 本身没有 mode 函数，用 `scipy.stats` 或者自己实现：

In [3]:
from scipy import stats

scores = np.array([85, 90, 85, 78, 92, 85, 88])
mode_result = stats.mode(scores)
print(f"众数: {mode_result.mode}, 出现次数: {mode_result.count}")  # 85, 3次


众数: 85, 出现次数: 3


> 💡 **小荷提醒**：选哪个指标要看场景。收入用中位数，考试分数用均值，类别数据用众数。

---

## 离散程度：数据有多"散"？

### 方差（var）和标准差（std）

衡量数据波动大小的核心指标。标准差 = 方差的平方根，**跟原始数据同单位**，更直观。

In [4]:
stock_a = np.array([10, 11, 9, 10, 10.5])   # 波动小
stock_b = np.array([5, 15, 8, 12, 20])      # 波动大

print(f"A股票: 均值={stock_a.mean():.2f}, 标准差={stock_a.std():.2f}")
# A股票: 均值=10.10, 标准差=0.67

print(f"B股票: 均值={stock_b.mean():.2f}, 标准差={5.29}")
# B股票: 均值=12.00, 标准差=5.29


A股票: 均值=10.10, 标准差=0.66
B股票: 均值=12.00, 标准差=5.29


**标准差越大，风险越高**——投资的时候这个指标很关键。

### 极差（ptp）和四分位距

In [5]:
data = np.array([23, 45, 56, 78, 89, 92, 95, 100])

# 极差 (peak to peak)
print(f"极差: {np.ptp(data)}")  # 100 - 23 = 77

# 四分位数
q1 = np.percentile(data, 25)   # 第一四分位数
q2 = np.percentile(data, 50)   # 中位数
q3 = np.percentile(data, 75)   # 第三四分位数
iqr = q3 - q1                   # 四分位距

print(f"Q1={q1}, Q2={q2}, Q3={q3}, IQR={iqr}")


极差: 77
Q1=53.25, Q2=83.5, Q3=92.75, IQR=39.5


> 四分位距（IQR）是判断异常值的利器：小于 Q1-1.5×IQR 或大于 Q3+1.5×IQR 的值，通常被认为是异常值。

---

## 实战：电商销售数据"体检"

假设你有一份 30 天的销售数据，来做个全面"体检"：

In [6]:
import numpy as np

# 模拟30天销售额（单位：万元）
np.random.seed(42)
sales = np.random.normal(50, 15, 30)  # 均值50，标准差15
sales[15] = 120  # 插入一个异常值（大促当天）

print("=" * 40)
print("📊 销售数据体检报告")
print("=" * 40)

# 基本统计
print(f"样本量: {len(sales)}")
print(f"均值: {sales.mean():.2f} 万元")
print(f"中位数: {np.median(sales):.2f} 万元")
print(f"标准差: {sales.std():.2f} 万元")
print(f"最小值: {sales.min():.2f} 万元")
print(f"最大值: {sales.max():.2f} 万元")
print(f"极差: {np.ptp(sales):.2f} 万元")

# 分位数
print(f"\n📈 分位数分布:")
for p in [10, 25, 50, 75, 90]:
    print(f"  P{p}: {np.percentile(sales, p):.2f} 万元")

# 变异系数（标准差/均值），衡量相对波动
cv = sales.std() / sales.mean()
print(f"\n📉 变异系数: {cv:.2%} (波动{'较大' if cv > 0.3 else '适中'})")


📊 销售数据体检报告
样本量: 30
均值: 49.79 万元
中位数: 46.55 万元
标准差: 18.58 万元
最小值: 21.30 万元
最大值: 120.00 万元
极差: 98.70 万元

📈 分位数分布:
  P10: 28.80 万元
  P25: 41.20 万元
  P50: 46.55 万元
  P75: 57.00 万元
  P90: 72.07 万元

📉 变异系数: 37.31% (波动较大)


输出示例：
```
========================================
📊 销售数据体检报告
========================================
样本量: 30
均值: 53.12 万元
中位数: 49.85 万元
标准差: 18.76 万元
最小值: 23.45 万元
最大值: 120.00 万元
极差: 96.55 万元

📈 分位数分布:
  P10: 32.15 万元
  P25: 40.28 万元
  P50: 49.85 万元
  P75: 63.42 万元
  P90: 78.91 万元

📉 变异系数: 35.30% (波动较大)
```

看到没？**均值比中位数高**，说明数据右偏（有异常高值）；**变异系数 35%**，说明波动不小。这些信息对业务决策很重要。

---

## 相关性分析：变量之间有关系吗？

### 相关系数（corrcoef）

衡量两个变量线性相关程度，范围 -1 到 1：

In [7]:
# 广告投入 vs 销售额
ad_spend = np.array([10, 15, 20, 25, 30, 35, 40])
sales = np.array([25, 35, 45, 50, 65, 70, 85])

corr_matrix = np.corrcoef(ad_spend, sales)
print(f"相关系数: {corr_matrix[0, 1]:.3f}")
# 相关系数: 0.989 (强正相关)


相关系数: 0.994


| 相关系数范围 | 相关程度 |
|-------------|---------|
| 0.0 ~ 0.3 | 弱相关 |
| 0.3 ~ 0.7 | 中等相关 |
| 0.7 ~ 1.0 | 强相关 |

> ⚠️ **注意**：相关不等于因果！广告和销量相关，但不一定是广告直接导致了销量。

---

## 多维数据的统计：axis 参数的使用

实际工作中数据往往是二维的（比如多行样本、多列特征），这时候要用 `axis` 参数指定统计方向：

In [8]:
# 3个班级，每个班5门课的成绩
scores = np.array([
    [85, 90, 78, 92, 88],  # 一班
    [70, 75, 80, 72, 78],  # 二班
    [88, 85, 90, 87, 91]   # 三班
])

# axis=0: 沿列统计（每门课跨班级的统计）
print("每门课的平均分:", scores.mean(axis=0))
# [81.  83.33 82.67 83.67 85.67]

# axis=1: 沿行统计（每个班的统计）
print("每个班的平均分:", scores.mean(axis=1))
# [86.6 75.  88.2]

# 每个班的总分
print("每个班总分:", scores.sum(axis=1))
# [433 375 441]

# 每门课的最高分
print("每门课最高分:", scores.max(axis=0))
# [88 90 90 92 91]


每门课的平均分: [81.         83.33333333 82.66666667 83.66666667 85.66666667]
每个班的平均分: [86.6 75.  88.2]
每个班总分: [433 375 441]
每门课最高分: [88 90 90 92 91]


**axis 记忆法**：
- `axis=0` = 纵向压扁 = 对每列做统计
- `axis=1` = 横向压扁 = 对每行做统计

---

## 常用统计函数速查表

| 函数 | 功能 | 示例 |
|------|------|------|
| `np.mean()` | 均值 | `arr.mean()` |
| `np.median()` | 中位数 | `np.median(arr)` |
| `np.std()` | 标准差 | `arr.std(ddof=1)` |
| `np.var()` | 方差 | `arr.var()` |
| `np.min()` / `np.max()` | 最值 | `arr.min(axis=0)` |
| `np.sum()` | 求和 | `arr.sum()` |
| `np.percentile()` | 分位数 | `np.percentile(arr, 75)` |
| `np.corrcoef()` | 相关系数 | `np.corrcoef(x, y)` |
| `np.argmax()` / `np.argmin()` | 最值索引 | `arr.argmax()` |
| `np.cumsum()` | 累积和 | `arr.cumsum()` |

---

## 本篇小结

统计函数是数据分析的"基本功"，NumPy 提供了一套高效易用的工具：

1. **集中趋势**：mean、median、mode 描述数据中心
2. **离散程度**：std、var、percentile 描述数据波动
3. **相关性**：corrcoef 分析变量关系
4. **多维统计**：axis 参数控制统计方向

记住萧何的智慧：**先摸清家底，才能运筹帷幄**。统计函数就是帮你"摸清家底"的工具。

---

## 课后练习

用下面这份数据，完成统计报告：

In [10]:
import numpy as np
np.random.seed(2024)

# 5个产品，12个月的销量数据
sales = np.random.poisson(lam=100, size=(5, 12))

# 任务1：计算每个产品的年均销量和标准差
# 任务2：找出销量波动最大的产品（变异系数最大）
# 任务3：计算每个产品Q1和Q4的销量相关系数


In [11]:
# 任务1：计算每个产品的年均销量和标准差
mean_sales = sales.mean(axis=1)     # 沿列方向（12个月）求均值 → shape (5,)
std_sales  = sales.std(axis=1)      # 同轴求标准差

for i, (m, s) in enumerate(zip(mean_sales, std_sales)):
    print(f"  产品{i+1}：均值 {m:.1f}，标准差 {s:.1f}")

  产品1：均值 101.3，标准差 9.8
  产品2：均值 100.9，标准差 7.7
  产品3：均值 101.2，标准差 7.7
  产品4：均值 102.2，标准差 9.3
  产品5：均值 102.8，标准差 9.9


In [12]:
# 任务2：找出销量波动最大的产品（变异系数最大）
cv = std_sales / mean_sales         # 变异系数，消除量纲影响
top_product = np.argmax(cv)
print(f"\n波动最大的产品：产品{top_product+1}（CV={cv[top_product]:.4f}）")


波动最大的产品：产品1（CV=0.0969）


In [13]:
# 任务3：计算每个产品Q1和Q4的销量相关系数
q1 = sales[:, 0:3]    # shape (5, 3)
q4 = sales[:, 9:12]   # shape (5, 3)

for i in range(5):
    r = np.corrcoef(q1[i], q4[i])[0, 1]   # corrcoef 返回 2×2 矩阵，取[0,1]
    print(f"  产品{i+1}：r = {r:.4f}")

  产品1：r = 0.8746
  产品2：r = 0.1785
  产品3：r = 0.9489
  产品4：r = 0.0476
  产品5：r = -0.9999


把你的代码和结论扔评论区，我看 👀

本篇完整代码包括练习题解答都已经上传至 GitHub 仓库，欢迎 Clone。

---

## 下期预告

> **第 6 篇：随机数与矩阵运算**
>
> 蒙特卡洛模拟怎么玩？矩阵乘法在推荐系统里有什么用？下篇带你看 NumPy 的进阶玩法，为后面的机器学习打基础。

---

**统计函数用熟了，数据分析就入门了一半。**

👇 点「在看」，推给更多想学数据分析的人  
💬 评论区说说你工作中最常用的统计指标是哪个  
⭐ 关注公众号，跟萧何迷妹一起进阶

---

*「数据分析从入门到精通」系列 · 第 5 篇*  
*上一篇：[第 4 篇：广播机制深入理解]*  
*下一篇：第 6 篇：随机数与矩阵运算*